### **✏️ Exercises**
- Use LangChain's docling integration for document ingestion
- Rewrite the whole pipeline using LangChain

## Integrate LangChain's Docling Loader

In [ ]:
# @title
!pip install -q --upgrade langchain-docling langchain-community langchain-text-splitters langchain-qdrant langchain-groq langchain-huggingface langchain

In [ ]:
from langchain_docling import DoclingLoader

FILE_PATH = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/AtliqAI_HR_Policies.pdf"

# Initialize the DoclingLoader
loader = DoclingLoader(file_path=FILE_PATH)

# Load the documents into LangChain format. The DoclingLoader directly returns
# a list of LangChain Document objects, which can be thought of as initial 'chunks'.
langchain_docs = loader.load()

print(f"Loaded {len(langchain_docs)} documents/initial chunks using LangChain's DoclingLoader.")
print("First document/chunk's page content sample:")
print(langchain_docs[0].page_content[:500])
print("\nFirst document/chunk's metadata:")
print(langchain_docs[0].metadata)

Now that the document is loaded as a LangChain `Document` object, you can further process it, for example, by splitting it into smaller chunks using a text splitter.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
langchain_chunks = text_splitter.split_documents(langchain_docs)

print(f"Split into {len(langchain_chunks)} chunks.")
print("First chunk's content sample:")
print(langchain_chunks[0].page_content[:500])

## LangChain-based RAG Pipeline

Now let's reconstruct the RAG pipeline using LangChain's abstractions.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

# Select your embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Initialize Qdrant Client
# Use location=":memory:" for instant testing
vectorstore = QdrantVectorStore.from_documents(
    documents=langchain_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="docling_rag"
)

print("LangChain Qdrant vector store initialized.")

Now that the vector store is set up, we can create a LangChain retriever and an LLM to build a RetrievalQA chain.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import os
from google.colab import userdata

# Safely extract your key from the Colab secrets manager
# Note: Ensure the secret name matches exactly what you typed on the left panel (e.g., "GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Define the GROQ model
GROQ_MODEL  = "openai/gpt-oss-safeguard-20b"

# 3. Create a LangChain retriever
langchain_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 4. Initialize the LangChain LLM (ChatGroq)
# Ensure GROQ_API_KEY is set in environment or Colab secrets
langchain_llm = ChatGroq(temperature=0.2, model_name=GROQ_MODEL)

# 6. Design your AI prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context.
Context: {context}
Question: {question}
""")

# 7. Construct LCEL pipeline
rag_chain = (
    {"context": langchain_retriever, "question": RunnablePassthrough()}

    | prompt
    | langchain_llm
    | StrOutputParser()
)

# 8. Run a live query
response = rag_chain.invoke("What are the key points?")
print(response)